# Session 2: BFS, Dijkstra and A* on a grid graph
**A. Real-world problem.** Real warehouse robots can move diagonally, and a diagonal move is
longer (sqrt 2 times) than a straight one. Counting steps (Session 1) is no longer enough:
we need the cheapest route. We also care how much work the search does, because a robot
that thinks for a long time wastes battery and time.

**Plan.** (1) turn the grid into a graph with edge costs 1 and sqrt 2, (2) write BFS and
Dijkstra, (3) add A* with an admissible heuristic, (4) check that A* finds the same cost as
Dijkstra while expanding fewer nodes, (5) study how aisle width and obstacle density change
the route and the work.

In [ ]:
import heapq
import math
import pathlib
from collections import deque

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
SQRT2 = math.sqrt(2.0)

## Step 1: the grid is a graph
A *node* is a free cell (column, row). An *edge* joins two neighbouring free cells and has a
cost. With 4-connection the neighbours are up, down, left, right (cost 1). With 8-connection
the four diagonal neighbours are added (cost sqrt 2 = 1.414...). A diagonal move is not
allowed to cut a corner: both cells beside it must be free.

In [ ]:
def neighbors(blocked, node, diag=False):
    """List of (neighbour, edge cost) for a node (col, row). blocked[row, col] is True for walls."""
    rows, cols = blocked.shape
    c, r = node
    out = []
    for dc, dr in ((1, 0), (-1, 0), (0, 1), (0, -1)):
        nc, nr = c + dc, r + dr
        if 0 <= nc < cols and 0 <= nr < rows and not blocked[nr, nc]:
            out.append(((nc, nr), 1.0))
    if diag:
        for dc, dr in ((1, 1), (1, -1), (-1, 1), (-1, -1)):
            nc, nr = c + dc, r + dr
            if (0 <= nc < cols and 0 <= nr < rows and not blocked[nr, nc]
                    and not blocked[r, nc] and not blocked[nr, c]):
                out.append(((nc, nr), SQRT2))
    return out


def from_text(lines):
    """Build (blocked, start, goal) from text rows: '#' wall, 'S' start, 'G' goal, '.' free."""
    rows = len(lines)
    cols = len(lines[0])
    blocked = np.zeros((rows, cols), dtype=bool)
    start = goal = None
    for r, line in enumerate(lines):
        for c, ch in enumerate(line):
            if ch == "#":
                blocked[r, c] = True
            elif ch == "S":
                start = (c, r)
            elif ch == "G":
                goal = (c, r)
    return blocked, start, goal


def path_cost(path):
    """Total cost of a path of neighbouring nodes: 1 per straight move, sqrt 2 per diagonal."""
    return sum(1.0 if (a[0] == b[0] or a[1] == b[1]) else SQRT2 for a, b in zip(path, path[1:]))


def _walk_back(parent, node):
    path = []
    while node is not None:
        path.append(node)
        node = parent[node]
    return path[::-1]

## Step 2: breadth-first search and Dijkstra
*Expanded nodes* = nodes taken from the waiting list and finished. It is our measure of work.

* **BFS** uses a first-in-first-out queue. It minimises the NUMBER of moves, so it is exactly
  right when every edge costs 1 (4-connection) but not when diagonals cost sqrt 2.
* **Dijkstra** keeps a priority queue (Python's `heapq`) ordered by cost-so-far g. It always
  finishes the cheapest unfinished node, so the first time it finishes the goal the cost is
  the minimum.

In [ ]:
def bfs(blocked, start, goal, diag=False):
    parent = {start: None}
    queue = deque([start])
    expanded = 0
    while queue:
        node = queue.popleft()
        expanded += 1
        if node == goal:
            path = _walk_back(parent, node)
            return {"path": path, "cost": path_cost(path), "expanded": expanded}
        for n, _ in neighbors(blocked, node, diag):
            if n not in parent:
                parent[n] = node
                queue.append(n)
    return {"path": None, "cost": math.inf, "expanded": expanded}


def dijkstra(blocked, start, goal, diag=False):
    g = {start: 0.0}
    parent = {start: None}
    heap = [(0.0, start)]
    done = set()
    while heap:
        cost, node = heapq.heappop(heap)
        if node in done:
            continue                      # an old, more expensive entry for this node
        done.add(node)
        if node == goal:
            return {"path": _walk_back(parent, node), "cost": cost, "expanded": len(done)}
        for n, w in neighbors(blocked, node, diag):
            new = cost + w
            if new < g.get(n, math.inf):
                g[n] = new
                parent[n] = node
                heapq.heappush(heap, (new, n))
    return {"path": None, "cost": math.inf, "expanded": len(done)}


def cost_map(blocked, source, diag=False):
    """Cheapest cost from `source` to every cell (inf where unreachable): Dijkstra without a goal."""
    dist = np.full(blocked.shape, math.inf)
    dist[source[1], source[0]] = 0.0
    heap = [(0.0, source)]
    while heap:
        cost, node = heapq.heappop(heap)
        if cost > dist[node[1], node[0]]:
            continue
        for n, w in neighbors(blocked, node, diag):
            if cost + w < dist[n[1], n[0]]:
                dist[n[1], n[0]] = cost + w
                heapq.heappush(heap, (cost + w, n))
    return dist

## Step 3: A* and an admissible heuristic
A* orders the priority queue by f = g + h, where g is the cost so far and h(n) is a guess of
the cost still to go. If h NEVER OVERESTIMATES the true remaining cost (h is *admissible*),
A* still returns the cheapest path. Why: when A* finishes the goal with cost C, every
unfinished node n on the true best path has f(n) = g(n) + h(n) <= g(n) + (true remaining
cost) = best cost, so a path cheaper than C would have been finished first.

* 4-connected grid: Manhattan distance dx + dy never overestimates (each move changes dx + dy by 1).
* 8-connected grid: octile distance (dx + dy) + (sqrt 2 - 2) * min(dx, dy) is the exact cost on an
  empty floor, so it never overestimates either.
Using h = 0 turns A* back into Dijkstra.

In [ ]:
def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def octile(a, b):
    dx, dy = abs(a[0] - b[0]), abs(a[1] - b[1])
    return (dx + dy) + (SQRT2 - 2.0) * min(dx, dy)


def euclidean(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])


def astar(blocked, start, goal, diag=False, h=None, weight=1.0):
    """A* search. h(node, goal) is the heuristic (default: octile if diag else Manhattan).
    weight > 1 inflates h; that breaks admissibility and the path may no longer be cheapest."""
    if h is None:
        h = octile if diag else manhattan
    g = {start: 0.0}
    parent = {start: None}
    heap = [(weight * h(start, goal), 0.0, start)]
    done = set()
    while heap:
        _, cost, node = heapq.heappop(heap)
        if node in done:
            continue
        done.add(node)
        if node == goal:
            return {"path": _walk_back(parent, node), "cost": cost, "expanded": len(done)}
        for n, w in neighbors(blocked, node, diag):
            new = cost + w
            if new < g.get(n, math.inf):
                g[n] = new
                parent[n] = node
                heapq.heappush(heap, (new + weight * h(n, goal), new, n))
    return {"path": None, "cost": math.inf, "expanded": len(done)}

## Step 4: warehouses to test on
* `random_grid`: obstacles with a given density, seeded so results repeat.
* `serpentine`: long shelf rows with an opening at alternating ends, so the robot must
  snake through the aisles. `aisle` is the aisle width in cells; shelves are 2 cells deep.

In [ ]:
# A small grid where BFS (fewest moves) is NOT the cheapest route when diagonals cost sqrt 2.
BFS_TRAP = ["S....#",
            "...#.G",
            "......",
            "#...#."]

# A 5 x 3 grid you can solve by hand (see the checkpoint): a wall in column 2 with its gap in row 2.
HAND_GRID = ["S.#..",
             "..#.G",
             "....."]


def random_grid(width, height, density, seed):
    """Random obstacles; the corners (0, 0) and (width-1, height-1) are kept free."""
    rng = np.random.default_rng(seed)
    blocked = rng.random((height, width)) < density
    blocked[0, 0] = False
    blocked[height - 1, width - 1] = False
    return blocked


def serpentine(width, height, aisle):
    """Shelf rows (2 cells deep) separated by aisles of `aisle` cells. Returns (blocked, start, goal, shelf_cells)."""
    blocked = np.zeros((height, width), dtype=bool)
    y, k = aisle, 0
    while y + 2 + aisle <= height:
        blocked[y:y + 2, :] = True
        if k % 2 == 0:
            blocked[y:y + 2, width - 1] = False      # opening at the right end
        else:
            blocked[y:y + 2, 0] = False              # opening at the left end
        y += 2 + aisle
        k += 1
    goal_col = width - 1 if k % 2 == 0 else 0
    return blocked, (0, 0), (goal_col, height - 1), int(blocked.sum())


def density_stats(density, n_seeds=40, width=30, height=20):
    """Average over seeded random grids (8-connected, corner to corner), solvable grids only."""
    costs, dij, ast = [], [], []
    for seed in range(n_seeds):
        b = random_grid(width, height, density, seed)
        rd = dijkstra(b, (0, 0), (width - 1, height - 1), True)
        if rd["path"] is None:
            continue
        costs.append(rd["cost"])
        dij.append(rd["expanded"])
        ast.append(astar(b, (0, 0), (width - 1, height - 1), True)["expanded"])
    return {"solvable": len(costs), "mean_cost": float(np.mean(costs)),
            "mean_dijkstra": float(np.mean(dij)), "mean_astar": float(np.mean(ast))}

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)

    # figure 1: the four searches on one warehouse
    rng_grid = random_grid(30, 20, 0.25, seed=0)
    start, goal = (0, 0), (29, 19)
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
    for ax, (name, fn) in zip(axes, [("Dijkstra", lambda b: dijkstra(b, start, goal, True)),
                                     ("A* (octile)", lambda b: astar(b, start, goal, True)),
                                     ("A* with h = 0", lambda b: astar(b, start, goal, True, h=lambda a, b2: 0.0))]):
        res = fn(rng_grid)
        ax.imshow(rng_grid, cmap="Greys", origin="upper")
        p = np.array(res["path"])
        ax.plot(p[:, 0], p[:, 1], "-", color="tab:blue", lw=2)
        ax.plot(*start, "gs")
        ax.plot(*goal, "r*", ms=11)
        ax.set_title(f"{name}\ncost {res['cost']:.2f}, expanded {res['expanded']}", fontsize=9)
        ax.set_xlabel("column (cells)")
    axes[0].set_ylabel("row (cells)")
    fig.tight_layout()
    fig.savefig(OUT / "search_compare.png", dpi=120)
    d, a = dijkstra(rng_grid, start, goal, True), astar(rng_grid, start, goal, True)
    print(f"random grid 25%: Dijkstra cost {d['cost']:.3f} expanded {d['expanded']}; A* cost {a['cost']:.3f} expanded {a['expanded']}")

    # figure 2: aisle width and obstacle density
    print("serpentine warehouse 30 x 25, 8-connected")
    widths = [1, 2, 3, 4, 5]
    lens, dij_exp, ast_exp, shelves = [], [], [], []
    for a_w in widths:
        b, s, g_, n_shelf = serpentine(30, 25, a_w)
        rd, ra = dijkstra(b, s, g_, True), astar(b, s, g_, True)
        lens.append(rd["cost"]); dij_exp.append(rd["expanded"]); ast_exp.append(ra["expanded"]); shelves.append(n_shelf)
        print(f"  aisle {a_w}: shelf cells {n_shelf:3d}, path cost {rd['cost']:.2f}, Dijkstra expanded {rd['expanded']}, A* expanded {ra['expanded']}")
    densities = [0.0, 0.1, 0.2, 0.3, 0.4]
    mean_cost, mean_dij, mean_ast = [], [], []
    for dens in densities:
        st = density_stats(dens)
        mean_cost.append(st["mean_cost"]); mean_dij.append(st["mean_dijkstra"]); mean_ast.append(st["mean_astar"])
        print(f"  density {dens:.1f}: solvable {st['solvable']:2d}/40, mean cost {st['mean_cost']:.2f}, mean expanded Dijkstra {st['mean_dijkstra']:.0f}, A* {st['mean_astar']:.0f}")
    fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.8))
    ax1.plot(widths, lens, "o-", label="path cost")
    ax1.set_xlabel("aisle width (cells)")
    ax1.set_ylabel("path cost (cell widths)")
    ax1.set_title("Narrow aisles: more shelves, longer route")
    ax1.grid(alpha=0.3)
    ax1.legend(fontsize=8)
    ax2.plot(densities, mean_dij, "o-", label="Dijkstra")
    ax2.plot(densities, mean_ast, "s-", label="A* (octile)")
    ax2.set_xlabel("obstacle density (fraction of cells)")
    ax2.set_ylabel("mean nodes expanded")
    ax2.set_title("Work versus obstacle density (30 x 20 grid)")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "aisle_density.png", dpi=120)

## Where this stops
Grid paths have jagged directions and only 8 headings. Session 3 leaves the grid: it plans
in continuous space with RRT and compares with A*.

## Coding tasks
1. **Easy.** Run `dijkstra` and `astar` on `random_grid(30, 20, 0.2, 5)` and print the cost and the expanded nodes of each.
2. **Stretch.** Call `astar` with `h=euclidean` on an 8-connected grid and compare its expanded nodes with octile. Is it still optimal? Explain.
3. **Explain.** Explain why a diagonal move must not cut a corner for a robot with a body, with a sketch.

## Assessment
- **A. Conceptual.** Why does an admissible heuristic keep A* optimal, and what goes wrong if h overestimates?
- **B. Numerical.** For a goal dx = 3 columns and dy = 1 row away on an empty 8-connected floor, compute the Manhattan, octile and straight-line distances and say which are admissible.
- **C. Graph and simulation interpretation.** In the density graph, how does the work of Dijkstra change as the density of obstacles grows, and why does A* expand fewer nodes at every density?
- **D. Design and optimisation challenge.** Design: in the 30 by 25 serpentine warehouse you need at least 200 shelf cells and a path cost of at most 170. Which aisle width (1 to 5 cells) do you choose, and what is the trade-off?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import math

import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    s2 = math.sqrt(2.0)

    # hand-checkable: empty 3 x 3 floor, corner to corner
    empty = np.zeros((3, 3), dtype=bool)
    checks.append(("3x3 empty, 4-connected: every method gives cost 4",
                   all(abs(f(empty, (0, 0), (2, 2), False)["cost"] - 4.0) < 1e-12 for f in (L.bfs, L.dijkstra, L.astar))))
    checks.append(("3x3 empty, 8-connected: Dijkstra and A* give 2 sqrt 2 = 2.828",
                   all(abs(f(empty, (0, 0), (2, 2), True)["cost"] - 2 * s2) < 1e-12 for f in (L.dijkstra, L.astar))))

    # worked example: the 5 x 3 hand grid
    blocked, start, goal = L.from_text(L.HAND_GRID)
    d4 = L.dijkstra(blocked, start, goal, False)
    d8 = L.dijkstra(blocked, start, goal, True)
    checks.append((f"worked example: 4-connected cost is 7 (got {d4['cost']:.3f})", abs(d4["cost"] - 7.0) < 1e-12))
    checks.append((f"worked example: 8-connected cost is 3 + 2 sqrt 2 = 5.828 (got {d8['cost']:.3f})", abs(d8["cost"] - (3 + 2 * s2)) < 1e-12))
    checks.append(("worked example: Manhattan lower bound is 5, less than 7", L.manhattan(start, goal) == 5 < d4["cost"]))
    checks.append(("worked example: A* agrees with Dijkstra in both modes",
                   abs(L.astar(blocked, start, goal, False)["cost"] - 7.0) < 1e-12
                   and abs(L.astar(blocked, start, goal, True)["cost"] - (3 + 2 * s2)) < 1e-12))
    # no corner cutting: (1,1) -> (2,2) is forbidden because (2,1) is a wall
    nb = [n for n, _ in L.neighbors(blocked, (1, 1), True)]
    checks.append(("a diagonal move may not cut a corner", (2, 2) not in nb and (2, 0) not in nb and (0, 0) in nb))
    checks.append(("path cost helper: 1 per straight move, sqrt 2 per diagonal", abs(L.path_cost([(0, 0), (1, 0), (2, 1)]) - (1 + s2)) < 1e-12))

    # BFS minimises moves, not cost, when diagonals are allowed
    tb, ts, tg = L.from_text(L.BFS_TRAP)
    bf, dj = L.bfs(tb, ts, tg, True), L.dijkstra(tb, ts, tg, True)
    checks.append((f"BFS trap grid: BFS costs 2 + 3 sqrt 2 = 6.243, Dijkstra 6.0 (got {bf['cost']:.3f}, {dj['cost']:.3f})",
                   abs(bf["cost"] - (2 + 3 * s2)) < 1e-9 and abs(dj["cost"] - 6.0) < 1e-9 and len(bf["path"]) < len(dj["path"])))
    checks.append(("with 4-connection BFS equals Dijkstra", abs(L.bfs(tb, ts, tg, False)["cost"] - L.dijkstra(tb, ts, tg, False)["cost"]) < 1e-12))

    # heuristics by hand
    checks.append(("octile((0,0),(3,1)) = 4 + (sqrt 2 - 2) * 1 = 2 + sqrt 2", abs(L.octile((0, 0), (3, 1)) - (2 + s2)) < 1e-12))
    checks.append(("Euclidean (0,0)-(3,1) = sqrt 10 = 3.162; Manhattan 4 exceeds the true 8-connected cost 3.414",
                   abs(L.euclidean((0, 0), (3, 1)) - math.sqrt(10)) < 1e-12 and round(L.euclidean((0, 0), (3, 1)), 3) == 3.162
                   and L.manhattan((0, 0), (3, 1)) > L.octile((0, 0), (3, 1)) and round(L.octile((0, 0), (3, 1)), 3) == 3.414))
    checks.append(("Manhattan((0,0),(3,1)) = 4", L.manhattan((0, 0), (3, 1)) == 4))

    # admissibility, numerically, on 25 random grids
    ok4 = ok8 = True
    for seed in range(25):
        b = L.random_grid(20, 14, 0.2, seed)
        goal_ = (19, 13)
        for diag, h in ((False, L.manhattan), (True, L.octile)):
            true = L.cost_map(b, goal_, diag)
            for r in range(14):
                for c in range(20):
                    if np.isfinite(true[r, c]) and h((c, r), goal_) > true[r, c] + 1e-9:
                        if diag:
                            ok8 = False
                        else:
                            ok4 = False
    checks.append(("Manhattan never overestimates on 4-connected grids (25 grids)", ok4))
    checks.append(("octile never overestimates on 8-connected grids (25 grids)", ok8))
    checks.append(("Manhattan DOES overestimate on an 8-connected grid: (0,0)->(1,1) is 2 but the true cost is sqrt 2",
                   L.manhattan((0, 0), (1, 1)) > L.cost_map(np.zeros((2, 2), dtype=bool), (1, 1), True)[0, 0]))

    # A* equals Dijkstra in cost and expands fewer nodes
    same, d_tot, a_tot, n = True, 0, 0, 0
    for seed in range(40):
        b = L.random_grid(30, 20, 0.2, seed)
        rd = L.dijkstra(b, (0, 0), (29, 19), True)
        ra = L.astar(b, (0, 0), (29, 19), True)
        if rd["path"] is None:
            same &= ra["path"] is None
            continue
        n += 1
        same &= abs(rd["cost"] - ra["cost"]) < 1e-9
        d_tot += rd["expanded"]
        a_tot += ra["expanded"]
    checks.append((f"A* and Dijkstra return the same cost on all {n} solvable 20%-density grids", same and n == 35))
    checks.append((f"A* expands fewer nodes in total ({a_tot} versus {d_tot})", a_tot < d_tot and d_tot == 16776 and a_tot == 6339))
    rg = L.random_grid(30, 20, 0.25, 0)
    rd, ra = L.dijkstra(rg, (0, 0), (29, 19), True), L.astar(rg, (0, 0), (29, 19), True)
    checks.append((f"25% grid, seed 0: cost 40.971 for both; expanded 463 versus 148 (got {rd['expanded']}, {ra['expanded']})",
                   abs(rd["cost"] - 40.971) < 5e-4 and abs(ra["cost"] - rd["cost"]) < 1e-9 and rd["expanded"] == 463 and ra["expanded"] == 148))
    ra0 = L.astar(rg, (0, 0), (29, 19), True, h=lambda a, b: 0.0)
    checks.append(("A* with h = 0 is Dijkstra: same expanded count", ra0["expanded"] == rd["expanded"]))
    rw = L.astar(rg, (0, 0), (29, 19), True, weight=3.0)
    checks.append(("an inflated (inadmissible) heuristic expands fewer nodes but may cost more", rw["expanded"] < ra["expanded"] and rw["cost"] >= rd["cost"] - 1e-9))

    # empty-floor facts used in the text
    e = np.zeros((20, 30), dtype=bool)
    de = L.dijkstra(e, (0, 0), (29, 19), True)
    checks.append((f"empty 30 x 20 floor, corner to corner: 19 diagonals + 10 straights = 36.87 (got {de['cost']:.2f})",
                   abs(de["cost"] - (19 * s2 + 10)) < 1e-9 and round(de["cost"], 2) == 36.87))

    # design table: serpentine warehouse 30 x 25
    exp = {1: (464, 285.00), 2: (290, 192.73), 3: (232, 161.97), 4: (174, 131.21), 5: (116, 100.46)}
    ok = True
    for a_w, (cells, cost) in exp.items():
        b, s, g, n_cells = L.serpentine(30, 25, a_w)
        r = L.dijkstra(b, s, g, True)
        ok &= n_cells == cells and abs(r["cost"] - cost) < 6e-3 and abs(L.astar(b, s, g, True)["cost"] - r["cost"]) < 1e-9
    checks.append(("serpentine 30 x 25: shelf cells 464/290/232/174/116 and costs 285.00/192.73/161.97/131.21/100.46 for aisles 1..5", ok))
    feasible = [a_w for a_w, (cells, cost) in exp.items() if cells >= 200 and cost <= 170]
    checks.append(("design answer: aisle 3 is the only width with at least 200 shelf cells and cost at most 170", feasible == [3]))
    b1, s1, g1, _ = L.serpentine(30, 25, 1)
    r1d, r1a = L.dijkstra(b1, s1, g1, True), L.astar(b1, s1, g1, True)
    checks.append(("aisle 1 is a single corridor: Dijkstra and A* both expand 286 nodes", r1d["expanded"] == r1a["expanded"] == 286))

    # density table
    st = {d: L.density_stats(d) for d in (0.0, 0.1, 0.2, 0.3, 0.4)}
    checks.append(("solvable grids out of 40 at densities 0, 0.1, 0.2, 0.3, 0.4: 40, 40, 35, 19, 1",
                   [st[d]["solvable"] for d in sorted(st)] == [40, 40, 35, 19, 1]))
    costs = [round(st[d]["mean_cost"], 2) for d in sorted(st)]
    checks.append((f"mean cost rises with density: 36.87, 37.82, 39.77, 43.76, 48.49 (got {costs})", costs == [36.87, 37.82, 39.77, 43.76, 48.49]))
    checks.append(("Dijkstra's mean expanded nodes: 600 with no obstacles, 188 at density 0.4",
                   round(st[0.0]["mean_dijkstra"]) == 600 and round(st[0.4]["mean_dijkstra"]) == 188))
    checks.append(("A* expands fewer nodes than Dijkstra at every density",
                   all(st[d]["mean_astar"] < st[d]["mean_dijkstra"] for d in st)))

    # edge cases
    sealed = np.zeros((3, 3), dtype=bool)
    sealed[:, 1] = True
    for f in (L.bfs, L.dijkstra, L.astar):
        if f(sealed, (0, 0), (2, 2), True)["path"] is not None:
            checks.append(("a sealed wall gives no path", False))
            break
    else:
        checks.append(("a sealed wall gives no path (path None, cost inf) in all three methods", True))
    one = L.astar(empty, (1, 1), (1, 1), True)
    checks.append(("start equals goal: cost 0, one expanded node", one["cost"] == 0.0 and one["expanded"] == 1))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
